In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import torchvision

def find_dirs(root):
    subdirs = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
    img_dir = [d for d in subdirs if "image" in d.lower()][0]
    mask_dir = [d for d in subdirs if "mask" in d.lower()][0]
    return img_dir, mask_dir


class SegmentationDataset(Dataset):
    def __init__(self, root_dir):
        image_dir, mask_dir = find_dirs(root_dir)

        self.image_dir = os.path.join(root_dir, image_dir)
        self.mask_dir = os.path.join(root_dir, mask_dir)

        self.images = sorted(os.listdir(self.image_dir))
        self.masks = sorted(os.listdir(self.mask_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.image_dir, self.images[idx])).convert("RGB")
        mask = Image.open(os.path.join(self.mask_dir, self.masks[idx]))

        img = torchvision.transforms.ToTensor()(img)
        mask = torch.tensor(mask, dtype=torch.int64)
        mask = remap_mask(mask)

        return img, mask


dataset = SegmentationDataset(path)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

images, masks = next(iter(loader))

fig, axes = plt.subplots(4, 2, figsize=(6, 8))
for i in range(4):
    axes[i, 0].imshow(images[i].permute(1, 2, 0))
    axes[i, 0].axis("off")
    axes[i, 1].imshow(masks[i], cmap="gray")
    axes[i, 1].axis("off")

plt.show()



In [ ]:
import torch
import segmentation_models_pytorch as smp

num_classes = 3

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=num_classes
)

model


In [ ]:
import torch

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            running_loss += loss.item()

    return running_loss / len(loader)


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 10
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, val_loader, criterion, device) if 'val_loader' in globals() else validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

plt.figure(figsize=(10,4))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.legend()
plt.show()


In [ ]:
import torch
import matplotlib.pyplot as plt

model.eval()

loader_vis = val_loader if 'val_loader' in globals() else test_loader
images, masks = next(iter(loader_vis))
images = images.to(device)

with torch.no_grad():
    outputs = model(images)
    preds = torch.argmax(outputs, dim=1).cpu()

images = images.cpu()
masks = masks.cpu()

n = min(4, images.size(0))
fig, axes = plt.subplots(n, 3, figsize=(10, 3*n))

for i in range(n):
    axes[i, 0].imshow(images[i].permute(1, 2, 0))
    axes[i, 0].axis("off")

    axes[i, 1].imshow(masks[i], cmap="gray")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(preds[i], cmap="gray")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()
